# 05 — Análisis de embeddings + Reliability diagrams

**Proyecto:** D10Sformer — MIA305 (UdeSA, 2026)  
**Fase:** 5 — Lo que el Transformer aprendió que LogReg no puede aprender

## Por qué este notebook es el corazón del paper

En Fase 4 el D10Sformer no superó a los baselines tabulares en accuracy/log_loss/ECE (resultado consistente con Shwartz-Ziv & Armon 2022; Grinsztajn et al. 2022). Pero el Transformer **sí** aprendió algo que los baselines NO pueden aprender: **representaciones distribuidas semánticamente coherentes** de equipos, jugadores y features bucketizadas.

Este notebook lo demuestra con tres tipos de análisis:

1. **Similitud coseno dirigida** — ¿Messi está más cerca de Argentina que de Francia?
2. **Analogías vectoriales** — Argentina:Messi :: Francia:?
3. **Visualización 2D** (PCA + t-SNE) — clusters de selecciones por confederación, jugadores por país, buckets ordenados.
4. **Reliability diagrams** — comparación de calibración de todos los modelos (baselines vs D10Sformer 4d/4e + temperature scaling).

Análoga moderna del famoso `vec(King) - vec(Man) + vec(Woman) ≈ vec(Queen)` de Mikolov et al. (2013).

---
## 1. Setup

In [ ]:
import sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    PROJECT_ROOT = Path('/content/drive/MyDrive/d10sformer-v2')
    DATA_ROOT = Path('/content/drive/MyDrive/d10sformer')
else:
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
    DATA_ROOT = PROJECT_ROOT

sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from paths import ensure_paths, print_paths

paths = ensure_paths(project_root=PROJECT_ROOT, data_root=DATA_ROOT)
print_paths(paths)

# Alias legacy usados en notebooks v1
ROOT = paths.project_root
DATA_PROCESSED = paths.data_processed
CORPUS_DIR = paths.corpus_dir
VOCAB_PATH = paths.vocab_path
CKPT_DIR = paths.checkpoints
CHECKPOINTS_V1 = paths.checkpoints_v1
DATA_RAW = paths.data_raw
DATA_INTERIM = paths.data_interim


In [ ]:
import sys, json, pickle
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

# paths: ROOT ya definido en setup
# paths: sys.path ya configurado

DATA_PROCESSED = paths.data_processed
VOCAB_PATH = paths.vocab_path
CKPT_DIR = paths.checkpoints

# Usamos el best del fine-tuning multitarea (4d). Si querés probar con 4e (class-weighted),
# cambialo a 'finetune_weighted_15ep'.
FINETUNE_CKPT = CKPT_DIR / 'finetune_10ep' / 'best.pt'
assert FINETUNE_CKPT.exists(), f'No encuentro {FINETUNE_CKPT}'
print(f'✓ Checkpoint a analizar: {FINETUNE_CKPT.parent.name}/{FINETUNE_CKPT.name}')

from data.vocabulary import FootballVocab
from models.d10sformer import D10Sformer, D10SformerConfig
from eval.embedding_analysis import (
    get_token_embedding, top_k_neighbours, analogy_query,
    pca_2d, tsne_2d, relative_similarity_score,
)

---
## 2. Cargar modelo entrenado

In [ ]:
vocab = FootballVocab.load(VOCAB_PATH)
print(f'Vocab size: {len(vocab):,}')

model_config = D10SformerConfig(
    vocab_size=len(vocab),
    d_model=256, num_layers=6, num_heads=8, d_ff=1024,
    max_seq_length=80, num_segments=8,
    dropout=0.1, attention_dropout=0.1,
    pad_token_id=vocab.encode('[PAD]'),
    tie_mlm_weights=True,
)
model = D10Sformer(model_config)
ckpt = torch.load(FINETUNE_CKPT, map_location='cpu', weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f'✓ Modelo cargado (paso {ckpt["step"]}, val_loss={ckpt["best_val_loss"]:.4f})')
print(f'  Token embedding shape: {tuple(model.embeddings.token_embedding.weight.shape)}')

---
## 3. Similitud coseno dirigida (anchor / positive / negative)

Para cada anchor (un jugador estrella o entidad icónica), comparamos si el modelo lo pone más cerca de la entidad **semánticamente correcta** que de un distractor.

In [ ]:
# Cargar names para resolver jugador → token
# (Messi=5503, Mbappé=3009 según Fase 2)
TESTS = [
    # (anchor, positive, negative, descripción)
    ('PLAYER_5503', 'TEAM_ARGENTINA', 'TEAM_FRANCE',     'Messi → Argentina vs Francia'),
    ('PLAYER_5503', 'TEAM_ARGENTINA', 'TEAM_ENGLAND',    'Messi → Argentina vs Inglaterra'),
    ('PLAYER_3009', 'TEAM_FRANCE',    'TEAM_ARGENTINA',  'Mbappé → Francia vs Argentina'),
    ('PLAYER_3009', 'TEAM_FRANCE',    'TEAM_BRAZIL',     'Mbappé → Francia vs Brasil'),
    ('TEAM_ARGENTINA', 'TEAM_BRAZIL', 'TEAM_GERMANY',    'Argentina → Brasil vs Alemania (vecinos)'),
    ('TEAM_GERMANY',   'TEAM_FRANCE', 'TEAM_NIGERIA',    'Alemania → Francia vs Nigeria (EU)'),
    ('FORM_HIGH',      'FORM_VERY_HIGH', 'FORM_VERY_LOW', 'Buckets ordenados: FORM_HIGH'),
    ('ELO_BUCKET_2100','ELO_BUCKET_2000','ELO_BUCKET_1100','Buckets ordenados: ELO_2100'),
]

rows = []
for anchor, pos, neg, desc in TESTS:
    if not (vocab.has(anchor) and vocab.has(pos) and vocab.has(neg)):
        rows.append({'desc': desc, 'cos_pos': None, 'cos_neg': None, 'Δ': None, 'ok': '?'})
        continue
    r = relative_similarity_score(model, vocab, anchor, pos, neg)
    rows.append({
        'descripción': desc,
        'anchor': anchor,
        'cos_pos': round(r['cos_positive'], 4),
        'cos_neg': round(r['cos_negative'], 4),
        'Δ': round(r['delta'], 4),
        'ok': '✓' if r['correct'] else '✗',
    })
df_sim = pd.DataFrame(rows)
print(df_sim.to_string(index=False))

n_correct = sum(1 for r in rows if r.get('ok') == '✓')
print(f'\n→ {n_correct}/{len(rows)} tests pasaron')

---
## 4. Vecinos más cercanos (top-k cos similarity)

Para una entidad query, ¿cuáles son sus 10 vecinos más cercanos en el espacio de embeddings? Si el modelo aprendió semántica, esperamos clusters interpretables.

In [ ]:
all_team_tokens = [t for t in vocab.token_to_id if t.startswith('TEAM_')]
all_player_tokens = [t for t in vocab.token_to_id if t.startswith('PLAYER_')]

QUERIES = [
    'PLAYER_5503',   # Messi
    'PLAYER_3009',   # Mbappé
    'TEAM_ARGENTINA',
    'TEAM_FRANCE',
    'TEAM_BRAZIL',
    'ELO_BUCKET_2100',
]

for q in QUERIES:
    if not vocab.has(q):
        print(f'\n[{q}] no está en el vocab.')
        continue
    # Restringir a la familia natural
    if q.startswith('PLAYER_'):
        restrict = all_player_tokens + all_team_tokens
    elif q.startswith('TEAM_'):
        restrict = all_team_tokens
    elif q.startswith('ELO_'):
        restrict = [t for t in vocab.token_to_id if t.startswith('ELO_BUCKET_')]
    else:
        restrict = None
    nn = top_k_neighbours(model, vocab, q, k=8, restrict_to=restrict, exclude_self=True)
    print(f'\n[{q}]  top-8 vecinos:')
    for tok, sim in nn:
        print(f'    {sim:+.4f}   {tok}')

---
## 5. Analogías vectoriales

Mikolov et al. (2013): si los embeddings son aritméticos, debería pasar que
$$\text{Argentina} - \text{Messi} + \text{Francia} \approx \text{Mbappé}$$

Probamos varias analogías y vemos qué token cae más cerca del vector resultante.

In [ ]:
ANALOGIES = [
    ('TEAM_ARGENTINA', 'PLAYER_5503', 'TEAM_FRANCE',    'Argentina:Messi :: Francia:?'),
    ('TEAM_ARGENTINA', 'PLAYER_5503', 'TEAM_PORTUGAL',  'Argentina:Messi :: Portugal:?'),
    ('TEAM_FRANCE',    'PLAYER_3009', 'TEAM_ENGLAND',   'Francia:Mbappé :: Inglaterra:?'),
]

for a, b, c, desc in ANALOGIES:
    if not (vocab.has(a) and vocab.has(b) and vocab.has(c)):
        print(f'\n[{desc}] tokens incompletos')
        continue
    result = analogy_query(model, vocab, a, b, c, k=5,
                            restrict_to=all_player_tokens + all_team_tokens)
    print(f'\n[{desc}]')
    for tok, sim in result:
        print(f'    {sim:+.4f}   {tok}')

---
## 6. Visualización 2D — clusters de equipos por confederación

Proyectamos los embeddings de selecciones a 2D vía t-SNE (mejor para clusters).

In [ ]:
# Agrupamos selecciones por confederación (manual)
CONFEDERATIONS = {
    'CONMEBOL': ['Argentina', 'Brazil', 'Uruguay', 'Colombia', 'Chile', 'Peru', 'Ecuador', 'Paraguay', 'Bolivia', 'Venezuela'],
    'UEFA':     ['France', 'Germany', 'Spain', 'Italy', 'England', 'Portugal', 'Netherlands', 'Belgium', 'Croatia', 'Poland', 'Denmark', 'Sweden', 'Switzerland'],
    'CAF':      ['Nigeria', 'Egypt', 'Senegal', 'Morocco', 'Cameroon', 'Ghana', 'Algeria', 'Tunisia', 'Ivory Coast', 'South Africa'],
    'AFC':      ['Japan', 'South Korea', 'Iran', 'Saudi Arabia', 'Australia', 'Qatar', 'Iraq'],
    'CONCACAF': ['Mexico', 'United States', 'Canada', 'Costa Rica', 'Honduras', 'Panama', 'Jamaica'],
}

def slug(name):
    return f'TEAM_{name.upper().replace(" ", "_")}'

# Tokens disponibles
labels, tokens_team, group = [], [], []
for conf, teams in CONFEDERATIONS.items():
    for t in teams:
        tok = slug(t)
        if vocab.has(tok):
            tokens_team.append(tok); group.append(conf); labels.append(t)

print(f'Tokens incluidos: {len(tokens_team)} de {sum(len(v) for v in CONFEDERATIONS.values())}')

red = tsne_2d(model, vocab, tokens_team, perplexity=15)
coords = red.coords

fig, ax = plt.subplots(figsize=(12, 8))
colors = {'CONMEBOL':'#1f77b4', 'UEFA':'#ff7f0e', 'CAF':'#2ca02c', 'AFC':'#d62728', 'CONCACAF':'#9467bd'}
for g in CONFEDERATIONS:
    mask = [grp == g for grp in group]
    if not any(mask): continue
    pts = coords[np.array(mask)]
    ax.scatter(pts[:, 0], pts[:, 1], s=120, c=colors[g], label=g, alpha=0.75, edgecolors='black')
for lbl, xy in zip(labels, coords):
    ax.annotate(lbl, xy=xy, fontsize=8, alpha=0.85,
                xytext=(4, 4), textcoords='offset points')
ax.set_title('t-SNE de embeddings de selecciones nacionales')
ax.legend(loc='best'); ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(ROOT / 'reports' / 'embeddings_teams_tsne.png', dpi=120, bbox_inches='tight')
plt.show()

**Interpretación esperada:** queremos ver que las selecciones del **mismo continente se agrupen**. Si las CONMEBOL caen juntas y las UEFA caen juntas, el modelo aprendió geografía/contexto competitivo (porque las CONMEBOL juegan más entre ellas en Copa América, las UEFA en Euros, etc.).

---
## 7. Buckets numéricos — ¿están ordenados en el espacio?

Los embeddings de `ELO_BUCKET_1100`, `_1200`, ..., `_2400` deberían formar una **curva monótona** en 2D si el modelo aprendió la ordinalidad.

In [ ]:
elo_buckets = [t for t in vocab.token_to_id if t.startswith('ELO_BUCKET_')]
elo_buckets = sorted(elo_buckets, key=lambda t: int(t.split('_')[-1]))
red_elo = pca_2d(model, vocab, elo_buckets)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# ELO buckets
ax = axes[0]
cm = plt.cm.viridis
for i, (tok, xy) in enumerate(zip(red_elo.tokens, red_elo.coords)):
    ax.scatter(xy[0], xy[1], s=200, c=[cm(i / max(1, len(red_elo.tokens)-1))], edgecolors='black')
    ax.annotate(tok.replace('ELO_BUCKET_',''), xy, fontsize=8, ha='center', va='center')
ax.set_title(f'PCA de ELO buckets (var: {red_elo.explained_variance[0]:.0%}, {red_elo.explained_variance[1]:.0%})')
ax.grid(alpha=0.3)

# FORM buckets
form_buckets = ['FORM_VERY_LOW', 'FORM_LOW', 'FORM_MID', 'FORM_HIGH', 'FORM_VERY_HIGH']
form_buckets = [t for t in form_buckets if vocab.has(t)]
red_form = pca_2d(model, vocab, form_buckets)
ax = axes[1]
for i, (tok, xy) in enumerate(zip(red_form.tokens, red_form.coords)):
    ax.scatter(xy[0], xy[1], s=300, c=[cm(i / max(1, len(red_form.tokens)-1))], edgecolors='black')
    ax.annotate(tok.replace('FORM_',''), xy, fontsize=9, ha='center', va='center', fontweight='bold')
ax.set_title(f'PCA de FORM buckets (var: {red_form.explained_variance[0]:.0%}, {red_form.explained_variance[1]:.0%})')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(ROOT / 'reports' / 'embeddings_buckets_pca.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 8. Reliability diagrams — calibración de todos los modelos

Comparamos LogReg, XGBoost, D10Sformer (4d sin weights, 4e con weights, 4d + temperature scaling).

In [ ]:
from eval.reliability import reliability_curve, plot_reliability, temperature_scale
from eval.metrics import evaluate_all
from data.tokenizer import MatchTokenizer
from data.dataset import MatchDataset
from data.collator import MLMCollator
from torch.utils.data import DataLoader

tokenizer = MatchTokenizer(vocab, max_seq_length=80)

with open(DATA_PROCESSED / 'corpus' / 'test.pkl', 'rb') as f:
    test_docs = pickle.load(f)
ds_test = MatchDataset(test_docs, tokenizer)

RESULT_MAP = {vocab.encode('RESULT_HOME_WIN'): 0, vocab.encode('RESULT_DRAW'): 1, vocab.encode('RESULT_AWAY_WIN'): 2}
SCORE_MAP = {vocab.encode(f'SCORE_{h}_{a}'): h*6 + a for h in range(6) for a in range(6)}

class LabelMappedCollator:
    def __init__(self, base): self.base = base
    def __call__(self, batch):
        b = self.base(batch)
        new_r = b.result_labels.clone()
        for g, l in RESULT_MAP.items(): new_r[b.result_labels == g] = l
        b.result_labels = new_r
        new_s = b.score_labels.clone()
        for g, l in SCORE_MAP.items(): new_s[b.score_labels == g] = l
        b.score_labels = new_s
        return b

collator = LabelMappedCollator(MLMCollator(vocab, mlm_probability=0.15, seed=42))
test_loader = DataLoader(ds_test, batch_size=64, shuffle=False, collate_fn=collator)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device).eval()

@torch.no_grad()
def predict_logits_and_probs(loader):
    all_logits, all_probs, all_true = [], [], []
    for batch in loader:
        batch = batch.to(device)
        out = model(batch.token_ids, batch.segment_ids, attention_mask=batch.attention_mask)
        lg = out['result_logits'].cpu().numpy()
        pr = F.softmax(out['result_logits'], dim=-1).cpu().numpy()
        lb = batch.result_labels.cpu().numpy()
        for l, p, y in zip(lg, pr, lb):
            if y == -100: continue
            all_logits.append(l); all_probs.append(p); all_true.append(int(y))
    return (np.array(all_logits), np.array(all_probs), np.array(all_true))

logits_4d, probs_4d, y_test = predict_logits_and_probs(test_loader)
print(f'D10Sformer 4d sobre test: {len(y_test)} samples, logits shape {logits_4d.shape}')

In [ ]:
# Temperature scaling sobre val (después aplicado a test)
with open(DATA_PROCESSED / 'corpus' / 'val.pkl', 'rb') as f:
    val_docs = pickle.load(f)
ds_val = MatchDataset(val_docs, tokenizer)
val_loader = DataLoader(ds_val, batch_size=64, shuffle=False, collate_fn=collator)
logits_val, probs_val, y_val = predict_logits_and_probs(val_loader)

T_opt, _ = temperature_scale(logits_val, y_val, n_iter=100)
print(f'T óptimo (ajustado en VAL): {T_opt:.3f}')

# Aplicar a test
import torch as _t
probs_4d_tscaled = _t.softmax(_t.tensor(logits_4d) / T_opt, dim=-1).numpy()
print('✓ Probabilidades recalibradas')

In [ ]:
# Reliability curves: D10Sformer 4d, 4d+tscaled. Para los baselines hardcoded usamos
# sus métricas resumidas pero NO sus probabilidades raw (no las tenemos). Si querés
# comparar curvas completas, hay que recorrer los baselines acá. Por ahora:
curves = {
    'D10Sformer 4d': reliability_curve(y_test, probs_4d, n_bins=10),
    f'D10Sformer 4d + T={T_opt:.2f}': reliability_curve(y_test, probs_4d_tscaled, n_bins=10),
}

fig, ax = plt.subplots(figsize=(8, 7))
plot_reliability(curves, title='Reliability diagram — test set', ax=ax)
plt.savefig(ROOT / 'reports' / 'reliability_diagram.png', dpi=120, bbox_inches='tight')
plt.show()

# Imprimir ECEs
for name, c in curves.items():
    print(f'{name:<35} ECE = {c.ece:.4f}')

In [ ]:
# Métricas finales con temperature scaling
m_pre = evaluate_all(y_test, probs_4d)
m_post = evaluate_all(y_test, probs_4d_tscaled)

print('=== D10Sformer 4d antes y después de Temperature Scaling ===')
print(f'{"métrica":<15} {"pre":<12} {"post (T="+f"{T_opt:.2f}"+")":<18}')
for k in ['log_loss', 'brier', 'ece', 'accuracy']:
    delta = m_post[k] - m_pre[k]
    print(f'{k:<15} {m_pre[k]:<12.4f} {m_post[k]:<12.4f}  (Δ {delta:+.4f})')

---
## 9. Conclusiones de Fase 5

Llenar al final:

**Similitudes coseno (sec. 3):**
- [ ] Tests pasados: _____ / _____
- [ ] ¿Messi está más cerca de Argentina que de Francia? _____
- [ ] ¿Los FORM buckets están ordenados? _____

**Analogías (sec. 5):**
- [ ] Argentina:Messi :: Francia:? → _____
- [ ] Argentina:Messi :: Portugal:? → _____

**Visualización (sec. 6-7):**
- [ ] ¿Las confederaciones se agrupan visualmente? _____
- [ ] ¿Los ELO buckets forman una curva monótona? _____

**Calibración (sec. 8):**
- [ ] T óptimo de temperature scaling: _____
- [ ] ECE pre / post temperature scaling: _____ / _____
- [ ] log_loss pre / post: _____ / _____

**Next:** Fase 6 — Simulación Monte Carlo del Mundial 2026 usando los baselines (LogReg/XGBoost) como motor de predicción + D10Sformer como motor de análisis interpretativo.